In [56]:
import os
print(os.getcwd())
print(os.listdir("."))

/content
['.config', 'infosys-franchise-ai', 'app', 'app_data', 'sample_data']


In [57]:
import os
print(os.listdir("app"))

['worker.py', 'static', 'config.py', 'domain_franchise.py', 'views_franchise.py', 'qa.py', 'server.py', 'rbac.py', 'seed_data.py', 'localuser.py', 'weather_context.py', 'blocks.py', 'views_core.py', 'live_engine.py', 'weather.py', 'scale_data.py', 'intent_router.py', 'analyst.py', 'gpu.py', 'admin_profile.py', 'db.py', 'entities.py', 'hardening.py', 'alts.py', 'mailer.py', 'seanav.py', '__pycache__']


In [58]:
!git clone https://github.com/NISHANTH-7929/infosys-franchise-ai.git
import os
print(os.listdir("infosys-franchise-ai"))

fatal: destination path 'infosys-franchise-ai' already exists and is not an empty directory.
['.gitignore', '.streamlit', '.agents', '.claude', 'LICENSE', 'README.md', 'Milestone2', '.git', 'Milestone1']


In [59]:
import os
print(os.listdir("infosys-franchise-ai/Milestone1"))
!cd infosys-franchise-ai && git branch -r

['.streamlit', 'tests', 'app.py', 'src', 'README.md', 'requirements.txt', 'screenshots', 'Milestone1_Authentication.ipynb']
  origin/HEAD -> origin/main
  origin/feature/copilot-fine-tuning-and-agents
  origin/m2/rbac
  origin/m2/ui
  origin/main


In [60]:
!cd infosys-franchise-ai && for b in m2/rbac m2/ui feature/copilot-fine-tuning-and-agents; do echo "== $b"; git ls-tree -r --name-only origin/$b | grep -E "live_engine|blocks|server.py"; done

== m2/rbac
Milestone2/app/blocks.py
Milestone2/app/live_engine.py
Milestone2/app/server.py
== m2/ui
== feature/copilot-fine-tuning-and-agents


In [61]:
!cd infosys-franchise-ai && git checkout -q m2/rbac
!cp infosys-franchise-ai/Milestone2/app/blocks.py app/
!cp infosys-franchise-ai/Milestone2/app/live_engine.py app/
!cp infosys-franchise-ai/Milestone2/app/server.py app/
import os
print(os.listdir("app"))

['worker.py', 'static', 'config.py', 'domain_franchise.py', 'views_franchise.py', 'qa.py', 'server.py', 'rbac.py', 'seed_data.py', 'localuser.py', 'weather_context.py', 'blocks.py', 'views_core.py', 'live_engine.py', 'weather.py', 'scale_data.py', 'intent_router.py', 'analyst.py', 'gpu.py', 'admin_profile.py', 'db.py', 'entities.py', 'hardening.py', 'alts.py', 'mailer.py', 'seanav.py', '__pycache__']


In [62]:
import sys
sys.path.insert(0, "/content/app")
import admin_profile as AP
print("Import worked!")

Import worked!


In [63]:
import os
print(os.path.getsize("/content/app/admin_profile.py"), "bytes")
print([n for n in dir(AP) if not n.startswith("__")])

22194 bytes
['B', 'LE', 'OUTLET_ROLES', 'PERMISSION_KEYS', 'PERMISSION_ROWS', 'ROLES', '_SAFE', '_admins_left', '_ago', '_cols', '_ensure', '_guard', '_need', '_outlets', '_ready', '_user', '_users', 'add_user', 'admin_page', 'base64', 'change_password', 'check_email', 'check_password', 'check_username', 'delete_user', 'hash_password', 'hashlib', 'hmac', 'init_admin_tables', 'latest_log', 'log_action', 'os', 'profile_page', 're', 'register', 'reset_password', 'secrets', 'set_lock', 'set_role', 'update_username', 'verify_password']


In [64]:
import os
print(os.path.getsize("/content/app/admin_profile.py"), "bytes")

22194 bytes


In [65]:
%%writefile /content/app/admin_profile.py
"""Task 3 - Admin dashboard + Profile page (Milestone 2, Member 3, branch m2/admin).

Everything for Steps 4 and 6 lives in this one file so it does not clash with the other members' files.
server.py only needs three tiny hooks:
    import admin_profile as AP
    V.PAGES["profile"] = AP.profile_page ; V.PAGES["admin"] = AP.admin_page   (+ menu entries)
    AP.register(app, user_of, ctx, need, ok, bad)

Temporary pieces (marked TEMP) exist only so Task 3 can be tested before Member 1 (auth) and Member 2 (RBAC)
are merged. After the merge, swap them for the team versions (see MERGE NOTES at the bottom).
"""
import os, re, hmac, base64, hashlib, secrets
import live_engine as LE
import blocks as B

ROLES = ["Admin", "Franchise Owner", "Store Manager", "Staff", "Auditor"]
OUTLET_ROLES = {"Store Manager", "Staff"}          # these roles are scoped to one outlet
_need = lambda c, cap: None                        # replaced by register() with server.need (Member 2 makes it raise 403)
_ready = {"ok": False}

# ------------------------------------------------------------------ password hashing
def hash_password(pw):
    """bcrypt when installed, otherwise PBKDF2 (both are salted, never store a plain password)."""
    try:
        import bcrypt
        return bcrypt.hashpw(pw.encode()[:72], bcrypt.gensalt()).decode()
    except ImportError:
        salt = os.urandom(16); it = 200_000
        dk = hashlib.pbkdf2_hmac("sha256", pw.encode(), salt, it)
        return "pbkdf2$%d$%s$%s" % (it, base64.b64encode(salt).decode(), base64.b64encode(dk).decode())

def verify_password(pw, stored):
    try:
        if stored.startswith("pbkdf2$"):
            _, it, salt, dk = stored.split("$")
            test = hashlib.pbkdf2_hmac("sha256", pw.encode(), base64.b64decode(salt), int(it))
            return hmac.compare_digest(test, base64.b64decode(dk))
        import bcrypt
        return bcrypt.checkpw(pw.encode()[:72], stored.encode())
    except Exception:
        return False

# ------------------------------------------------------------------ Milestone 1 validation rules
def check_email(e):
    e = (e or "").strip()
    m = re.fullmatch(r"([A-Za-z0-9._%+-]+)@([A-Za-z0-9-]+)\.([A-Za-z]{2,}(?:\.[A-Za-z]{2,})*)", e)
    if not m: return "Enter a valid email like name@example.com"
    if len(re.findall(r"[A-Za-z]", m.group(1))) < 2: return "Email needs at least 2 letters before @"
    if len(re.findall(r"[A-Za-z]", m.group(2))) < 2: return "Email needs at least 2 letters between @ and the dot"
    return None

def check_username(u):
    if not re.fullmatch(r"[A-Za-z0-9_. ]{3,30}", (u or "").strip()): return "Username must be 3-30 characters (letters, numbers, space, . or _)"
    return None

def check_password(p):
    p = p or ""
    if len(p) < 8: return "Password needs at least 8 characters"
    if not re.search(r"[A-Z]", p): return "Password needs an upper case letter"
    if not re.search(r"[a-z]", p): return "Password needs a lower case letter"
    if not re.search(r"\d", p): return "Password needs a number"
    if not re.search(r"[^A-Za-z0-9]", p): return "Password needs a special character"
    return None

# ------------------------------------------------------------------ database
def _cols():
    with LE._DB_LOCK, LE.conn() as c: return {r[1] for r in c.execute("PRAGMA table_info(users)")}

def init_admin_tables():
    """Create activity_log, create users if Member 1 has not yet, and add any missing column this task needs."""
    with LE._DB_LOCK, LE.conn() as c:
        c.execute("""CREATE TABLE IF NOT EXISTS users(id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT UNIQUE NOT NULL, username TEXT NOT NULL,
            password_hash TEXT NOT NULL, role TEXT NOT NULL DEFAULT 'Staff', outlet TEXT, locked INTEGER NOT NULL DEFAULT 0, created_at TEXT, last_login TEXT)""")
        have = {r[1] for r in c.execute("PRAGMA table_info(users)")}
        for col, ddl in (("outlet", "TEXT"), ("locked", "INTEGER NOT NULL DEFAULT 0"), ("created_at", "TEXT"), ("last_login", "TEXT")):
            if col not in have: c.execute(f"ALTER TABLE users ADD COLUMN {col} {ddl}")
        c.execute("""CREATE TABLE IF NOT EXISTS activity_log(id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, actor TEXT, action TEXT, target TEXT, detail TEXT)""")
        # TEMP: make sure one Admin row exists so Task 3 can be tested before Member 1's login seeds the real first Admin.
        if not c.execute("SELECT 1 FROM users WHERE role='Admin'").fetchone():
            c.execute("INSERT INTO users(email,username,password_hash,role,locked,created_at) VALUES(?,?,?,?,0,?)",
                      ("admin@local", "Admin", hash_password(os.environ.get("ADMIN_PASSWORD") or secrets.token_urlsafe(18)), "Admin", LE.now()))
    _ready["ok"] = True

def _ensure():
    if not _ready["ok"]: init_admin_tables()

def log_action(actor, action, target="", detail=""):
    with LE._DB_LOCK, LE.conn() as c:
        c.execute("INSERT INTO activity_log(ts,actor,action,target,detail) VALUES(?,?,?,?,?)", (LE.now(), str(actor), action, str(target), str(detail)[:300]))

def latest_log(n=50, actor=None):
    """Newest n log rows; pass actor to see only one person's actions."""
    _ensure(); q = "SELECT ts,actor,action,target,detail FROM activity_log" + (" WHERE lower(actor)=lower(?)" if actor else "") + " ORDER BY id DESC LIMIT ?"
    with LE._DB_LOCK, LE.conn() as c: rows = c.execute(q, ((actor, n) if actor else (n,))).fetchall()
    return [dict(zip(("ts", "actor", "action", "target", "detail"), r)) for r in rows]

_SAFE = "email, username, role, outlet, locked, created_at, last_login"      # never select password_hash for display
def _user(email):
    with LE._DB_LOCK, LE.conn() as c: r = c.execute(f"SELECT {_SAFE} FROM users WHERE lower(email)=lower(?)", (email or "",)).fetchone()
    return dict(zip(("email", "username", "role", "outlet", "locked", "created_at", "last_login"), r)) if r else None

def _users():
    with LE._DB_LOCK, LE.conn() as c: rows = c.execute(f"SELECT {_SAFE} FROM users ORDER BY created_at, email").fetchall()
    return [dict(zip(("email", "username", "role", "outlet", "locked", "created_at", "last_login"), r)) for r in rows]

def _admins_left(excluding, active_only=False):
    """How many Admins would remain if `excluding` were removed (or locked, when active_only)."""
    q = "SELECT COUNT(*) FROM users WHERE role='Admin' AND lower(email)<>lower(?)" + (" AND locked=0" if active_only else "")
    with LE._DB_LOCK, LE.conn() as c: return c.execute(q, (excluding,)).fetchone()[0]

def _outlets():
    try: return B.Q("SELECT outlet_id FROM outlets ORDER BY outlet_id").outlet_id.tolist()
    except Exception: return []

# ------------------------------------------------------------------ actions (raise ValueError with a clear message on a broken rule)
def update_username(email, new):
    _ensure(); new = (new or "").strip(); err = check_username(new)
    if err: raise ValueError(err)
    with LE._DB_LOCK, LE.conn() as c:
        if c.execute("SELECT 1 FROM users WHERE lower(username)=lower(?) AND lower(email)<>lower(?)", (new, email)).fetchone(): raise ValueError("That username is already taken")
        if not c.execute("UPDATE users SET username=? WHERE lower(email)=lower(?)", (new, email)).rowcount: raise ValueError("Account not found")
    log_action(email, "profile_username", email, new); return "Username updated"

def change_password(email, current, new, confirm):
    _ensure()
    with LE._DB_LOCK, LE.conn() as c: r = c.execute("SELECT password_hash FROM users WHERE lower(email)=lower(?)", (email,)).fetchone()
    if not r: raise ValueError("Account not found")
    if not verify_password(current or "", r[0]): raise ValueError("Current password is wrong")
    err = check_password(new)
    if err: raise ValueError(err)
    if new != confirm: raise ValueError("New password and confirm password do not match")
    if verify_password(new, r[0]): raise ValueError("Choose a password different from the current one")
    with LE._DB_LOCK, LE.conn() as c: c.execute("UPDATE users SET password_hash=? WHERE lower(email)=lower(?)", (hash_password(new), email))
    log_action(email, "profile_password", email, "password changed"); return "Password changed"

def add_user(actor, email, username, password, role, outlet):
    _ensure(); email = (email or "").strip().lower(); username = (username or "").strip(); outlet = None if outlet in (None, "", "-") else outlet
    for err in (check_email(email), check_username(username), check_password(password)):
        if err: raise ValueError(err)
    if role not in ROLES: raise ValueError("Choose a valid role")
    if role in OUTLET_ROLES and not outlet: raise ValueError(f"{role} must be assigned to an outlet")
    if role not in OUTLET_ROLES: outlet = None
    with LE._DB_LOCK, LE.conn() as c:
        if c.execute("SELECT 1 FROM users WHERE lower(email)=lower(?)", (email,)).fetchone(): raise ValueError("A user with this email already exists")
        if c.execute("SELECT 1 FROM users WHERE lower(username)=lower(?)", (username,)).fetchone(): raise ValueError("That username is already taken")
        c.execute("INSERT INTO users(email,username,password_hash,role,outlet,locked,created_at) VALUES(?,?,?,?,?,0,?)", (email, username, hash_password(password), role, outlet, LE.now()))
    log_action(actor, "add_user", email, f"role={role}" + (f", outlet={outlet}" if outlet else "")); return f"User {email} added as {role}"

def set_role(actor, email, role, outlet=None):
    _ensure(); u = _user(email)
    if not u: raise ValueError("User not found")
    if role not in ROLES: raise ValueError("Choose a valid role")
    if u["email"].lower() == (actor or "").lower(): raise ValueError("You cannot change your own role")
    if u["role"] == "Admin" and role != "Admin" and _admins_left(u["email"], active_only=True) == 0: raise ValueError("The last remaining active Admin cannot be demoted")
    outlet = None if outlet in (None, "", "-") else outlet
    if role in OUTLET_ROLES:
        outlet = outlet or u["outlet"]
        if not outlet: raise ValueError(f"{role} needs an outlet: pick one in the Outlet box")
    else: outlet = None
    with LE._DB_LOCK, LE.conn() as c: c.execute("UPDATE users SET role=?, outlet=? WHERE lower(email)=lower(?)", (role, outlet, u["email"]))
    log_action(actor, "change_role", u["email"], f"{u['role']} -> {role}"); return f"{u['email']} is now {role}"

def set_lock(actor, email, lock):
    _ensure(); u = _user(email)
    if not u: raise ValueError("User not found")
    if lock:
        if u["email"].lower() == (actor or "").lower(): raise ValueError("You cannot lock your own account")
        if u["role"] == "Admin" and _admins_left(u["email"], active_only=True) == 0: raise ValueError("The last active Admin cannot be locked")
    sets = "locked=?" + (", failed_attempts=0" if (not lock and "failed_attempts" in _cols()) else "")
    with LE._DB_LOCK, LE.conn() as c: c.execute(f"UPDATE users SET {sets} WHERE lower(email)=lower(?)", (1 if lock else 0, u["email"]))
    log_action(actor, "lock_user" if lock else "unlock_user", u["email"]); return f"{u['email']} {'locked' if lock else 'unlocked'}"

def reset_password(actor, email, temp):
    """Admin sets a new temporary password for a user (for example someone who forgot it)."""
    _ensure(); u = _user(email)
    if not u: raise ValueError("User not found")
    err = check_password(temp)
    if err: raise ValueError(err)
    with LE._DB_LOCK, LE.conn() as c: c.execute("UPDATE users SET password_hash=? WHERE lower(email)=lower(?)", (hash_password(temp), u["email"]))
    log_action(actor, "reset_password", u["email"], "temporary password set"); return f"Temporary password set for {u['email']}"

def delete_user(actor, email, confirm):
    _ensure(); u = _user(email)
    if not u: raise ValueError("User not found")
    if u["email"].lower() == (actor or "").lower(): raise ValueError("You cannot delete your own account")
    if u["role"] == "Admin" and _admins_left(u["email"], active_only=True) == 0: raise ValueError("The last remaining active Admin cannot be deleted")
    if (confirm or "").strip() != "DELETE": raise ValueError("Type DELETE in the confirmation box to delete this user")
    with LE._DB_LOCK, LE.conn() as c: c.execute("DELETE FROM users WHERE lower(email)=lower(?)", (u["email"],))
    log_action(actor, "delete_user", u["email"], f"role={u['role']}"); return f"User {u['email']} deleted"

# ------------------------------------------------------------------ pages (server-driven blocks)
def _guard(c):
    """Admin pages and routes: need(ctx,'users') (Member 2 makes it raise 403) plus a hard Admin check as a second lock."""
    _need(c, "users")
    if c.get("role") != "Admin":
        from fastapi import HTTPException
        raise HTTPException(403, "Only the Admin role can open this page")

def _ago(ts): return (ts or "–").replace("T", " ")[:16]

def profile_page(c, tab=None):
    _ensure(); u = _user(c.get("email")) or {"email": c.get("email"), "username": (c.get("user") or {}).get("username", ""), "role": c.get("role"), "outlet": None, "created_at": None, "last_login": None}
    info = [{"field": "Username", "value": u["username"]}, {"field": "Email", "value": u["email"]}, {"field": "Role", "value": u["role"]},
            {"field": "Outlet", "value": u["outlet"] or "All / not assigned"}, {"field": "Joined", "value": _ago(u["created_at"])}, {"field": "Last login", "value": _ago(u["last_login"])}]
    bl = [B.kpis(("Role", u["role"]), ("Joined", (u["created_at"] or "–")[:10]), ("Last login", _ago(u["last_login"]))),
          B.table(info, ["field", "value"], title="Account details", sub="Your password is stored as a hash and is never shown.", key="pf_info"),
          B.row(B.form("pf_name", [{"name": "username", "label": "New username", "type": "text", "value": u["username"]}], "Save username", "/api/profile/username", title="Change username"),
                B.form("pf_pass", [{"name": "current", "label": "Current password", "type": "password"}, {"name": "new", "label": "New password", "type": "password", "placeholder": "8+ chars, Aa, 1, #"},
                                   {"name": "confirm", "label": "Confirm new password", "type": "password"}], "Change password", "/api/profile/password", title="Change password",
                       sub="Needs 8+ characters with upper case, lower case, a number and a special character.")),
          B.table(latest_log(10, c.get("email")), ["ts", "action", "target", "detail"], title="My recent activity", sub="The last 10 things you did in this app (username and password changes, admin actions).", key="pf_log")]
    return B.page("Profile", "Your account", bl, live=0)

PERMISSION_ROWS = [("Admin", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓"), ("Franchise Owner", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✗"),
                   ("Store Manager", "✓", "✓", "✓", "✓", "✗", "✓", "✓", "✗", "✗"), ("Staff", "✓", "✗", "✗", "✗", "✗", "✗", "✗", "✗", "✗"), ("Auditor", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✗", "✗")]
PERMISSION_KEYS = ["role", "copilot", "workforce", "outlets", "inventory", "marketing", "sentiment", "audit", "run_agents", "users"]

def admin_page(c, tab=None):
    _guard(c); _ensure(); us = _users(); emails = [u["email"] for u in us]; outs = ["-"] + _outlets()
    per = {r: sum(1 for u in us if u["role"] == r) for r in ROLES}; nl = sum(1 for u in us if u["locked"])
    rows = [{"email": u["email"], "username": u["username"], "role": u["role"], "outlet": u["outlet"] or "–", "status": "Locked" if u["locked"] else "Active", "created": _ago(u["created_at"]), "last_login": _ago(u["last_login"])} for u in us]
    first = emails[0] if emails else ""
    never = sum(1 for u in us if not u["last_login"])
    by_outlet = {}
    for u in us:
        if u["outlet"]: by_outlet[u["outlet"]] = by_outlet.get(u["outlet"], 0) + 1
    acts = {}
    for r in latest_log(500): acts[r["action"]] = acts.get(r["action"], 0) + 1
    bl = [B.kpis(("Total users", len(us)), ("Locked users", nl, None, "Critical" if nl else None), ("Never logged in", never, "no login yet", "Warning" if never else None), *[(r, per[r]) for r in ROLES]),
          B.row(B.card("Users by role", B.chart(B.pie({r: n for r, n in per.items() if n}), h=230, key="adm_pie_role")),
                B.card("Active vs locked", B.chart(B.pie({"Active": len(us) - nl, "Locked": nl}, colors={"Active": B.TEAL, "Locked": B.RED}), h=230, key="adm_pie_lock"))),
          B.row(B.card("Users per outlet", B.chart(B.bar(list(by_outlet) or ["none"], list(by_outlet.values()) or [0], colors=[B.BLUE]), h=230, key="adm_bar_outlet")),
                B.card("Admin actions by type", B.chart(B.bar(list(acts) or ["none"], list(acts.values()) or [0], horizontal=True, colors=[B.VIOLET]), h=230, key="adm_bar_act"))),
          B.table(rows, ["email", "username", "role", "outlet", B.pill_col("status"), "created", "last_login"], title="Users", sub="Passwords and hashes are never shown.", search=True, h=330, key="adm_users"),
          B.form("adm_add", [{"name": "email", "label": "Email", "type": "text"}, {"name": "username", "label": "Username", "type": "text"},
                             {"name": "password", "label": "Temporary password", "type": "password"}, {"name": "role", "label": "Role", "type": "select", "options": ROLES, "value": "Staff"},
                             {"name": "outlet", "label": "Outlet (Store Manager / Staff)", "type": "select", "options": outs, "value": "-"}], "Add user", "/api/admin/add_user", title="Add user",
                 sub="Same rules as signup: email 2/2/2 letters, strong password. Store Manager and Staff need an outlet."),
          B.row(B.form("adm_role", [{"name": "email", "label": "User", "type": "select", "options": emails, "value": first}, {"name": "role", "label": "New role", "type": "select", "options": ROLES, "value": "Staff"},
                                    {"name": "outlet", "label": "Outlet (if needed)", "type": "select", "options": outs, "value": "-"}], "Change role", "/api/admin/role", title="Promote / demote"),
                B.form("adm_lock", [{"name": "email", "label": "User", "type": "select", "options": emails, "value": first}, {"name": "action", "label": "Action", "type": "select", "options": ["Lock", "Unlock"], "value": "Lock"}],
                       "Apply", "/api/admin/lock", title="Lock / Unlock", tone="primary", sub="A locked user is blocked on the very next request."),
                B.form("adm_del", [{"name": "email", "label": "User", "type": "select", "options": emails, "value": first}, {"name": "confirm", "label": "Type DELETE to confirm", "type": "text"}],
                       "Delete user", "/api/admin/delete", title="Delete user", tone="danger", sub="You cannot delete yourself or the last Admin."),
                B.form("adm_reset", [{"name": "email", "label": "User", "type": "select", "options": emails, "value": first}, {"name": "temp", "label": "Temporary password", "type": "password"}],
                       "Reset password", "/api/admin/reset_password", title="Reset password", sub="Sets a new temporary password (strong password rules apply).")),
          B.table(latest_log(50), ["ts", "actor", "action", "target", "detail"], title="Activity log (latest 50)", sub="Search by person, action or user.", search=True, h=320, key="adm_log"),
          B.table([dict(zip(PERMISSION_KEYS, r)) for r in PERMISSION_ROWS], PERMISSION_KEYS, title="Role permission table", sub="Run agents = can start agent cycles. Users = can open this Admin page.", key="adm_perm")]
    return B.page("Admin", "Users, roles and activity", bl, live=0)

# ------------------------------------------------------------------ routes
def register(app, user_of, ctx, need, ok, bad):
    """Call once from server.py after user_of / ctx / need / ok / bad are defined."""
    global _need; _need = need
    from fastapi import Request

    @app.on_event("startup")
    def _admin_startup(): init_admin_tables()

    def run(fn, *a):
        try: return ok(fn(*a), refresh=True)
        except ValueError as e: return bad(str(e))

    # profile: any signed-in user, own account only
    @app.post("/api/profile/username")
    async def pf_username(request: Request):
        u = user_of(request); d = await request.json(); return run(update_username, u["email"], d.get("username"))
    @app.post("/api/profile/password")
    async def pf_password(request: Request):
        u = user_of(request); d = await request.json(); return run(change_password, u["email"], d.get("current"), d.get("new"), d.get("confirm"))

    # admin: every route is behind need(ctx, "users")
    async def admin_body(request):
        u = user_of(request); c = ctx(u); _guard(c); return u, await request.json()
    @app.post("/api/admin/add_user")
    async def ad_add(request: Request):
        u, d = await admin_body(request); return run(add_user, u["email"], d.get("email"), d.get("username"), d.get("password"), d.get("role"), d.get("outlet"))
    @app.post("/api/admin/role")
    async def ad_role(request: Request):
        u, d = await admin_body(request); return run(set_role, u["email"], d.get("email"), d.get("role"), d.get("outlet"))
    @app.post("/api/admin/lock")
    async def ad_lock(request: Request):
        u, d = await admin_body(request); return run(set_lock, u["email"], d.get("email"), d.get("action") == "Lock")
    @app.post("/api/admin/reset_password")
    async def ad_reset(request: Request):
        u, d = await admin_body(request); return run(reset_password, u["email"], d.get("email"), d.get("temp"))
    @app.post("/api/admin/delete")
    async def ad_delete(request: Request):
        u, d = await admin_body(request); return run(delete_user, u["email"], d.get("email"), d.get("confirm"))

# MERGE NOTES
#  * Member 1 (auth): user_of() must return the DB user and 401/403 for unknown or locked users. Then remove the TEMP admin seed above.
#  * Member 2 (rbac): ROLE_MENU needs "profile" for every role and "admin" for Admin only; ROLE_CAPS needs "users" for Admin only.

Overwriting /content/app/admin_profile.py


In [66]:
import os
print(os.path.getsize("/content/app/admin_profile.py"), "bytes")

22194 bytes


In [67]:
!cd infosys-franchise-ai && git ls-tree -r --name-only origin/m2/rbac | grep -E "^Milestone2/" | grep -v -E "screenshots|\.png|\.ipynb"

Milestone2/.gitignore
Milestone2/app/alts.py
Milestone2/app/analyst.py
Milestone2/app/blocks.py
Milestone2/app/config.py
Milestone2/app/db.py
Milestone2/app/domain_franchise.py
Milestone2/app/entities.py
Milestone2/app/gpu.py
Milestone2/app/hardening.py
Milestone2/app/intent_router.py
Milestone2/app/live_engine.py
Milestone2/app/localuser.py
Milestone2/app/mailer.py
Milestone2/app/qa.py
Milestone2/app/rbac.py
Milestone2/app/scale_data.py
Milestone2/app/seanav.py
Milestone2/app/seed_data.py
Milestone2/app/server.py
Milestone2/app/static/app.js
Milestone2/app/static/index.html
Milestone2/app/static/style.css
Milestone2/app/static/vendor/echarts.min.js
Milestone2/app/static/vendor/fonts/inter-latin-ext-wght-normal.woff2
Milestone2/app/static/vendor/fonts/inter-latin-wght-normal.woff2
Milestone2/app/static/vendor/leaflet.css
Milestone2/app/static/vendor/leaflet.js
Milestone2/app/static/vendor/topojson-client.min.js
Milestone2/app/static/vendor/world.topo.json
Milestone2/app/views_core.py
M

In [68]:
!cp -r infosys-franchise-ai/Milestone2/app/. /content/app/
import os
print(sorted(f for f in os.listdir("/content/app") if f.endswith(".py")))
print(os.path.getsize("/content/app/admin_profile.py"), "bytes")

['admin_profile.py', 'alts.py', 'analyst.py', 'blocks.py', 'config.py', 'db.py', 'domain_franchise.py', 'entities.py', 'gpu.py', 'hardening.py', 'intent_router.py', 'live_engine.py', 'localuser.py', 'mailer.py', 'qa.py', 'rbac.py', 'scale_data.py', 'seanav.py', 'seed_data.py', 'server.py', 'views_core.py', 'views_franchise.py', 'weather.py', 'weather_context.py', 'worker.py']
22194 bytes


In [69]:
import sys, importlib
sys.path.insert(0, "/content/app")
import admin_profile as AP
AP = importlib.reload(AP)

AP.init_admin_tables()
print(AP._users())
print(AP.add_user("admin@local", "test.user@example.com", "TestUser", "Test@1234", "Auditor", None))
print([u["email"] for u in AP._users()])
print(AP.latest_log(3))

[{'email': 'admin@local', 'username': 'Admin', 'role': 'Admin', 'outlet': None, 'locked': 0, 'created_at': '2026-11-01T10:10:10', 'last_login': None}]
User test.user@example.com added as Auditor
['admin@local', 'test.user@example.com']
[{'ts': '2026-11-01T10:16:40', 'actor': 'admin@local', 'action': 'add_user', 'target': 'test.user@example.com', 'detail': 'role=Auditor'}, {'ts': '2026-11-01T10:10:42', 'actor': 'admin@local', 'action': 'delete_user', 'target': 'test.user@example.com', 'detail': 'role=Auditor'}, {'ts': '2026-11-01T10:10:42', 'actor': 'admin@local', 'action': 'unlock_user', 'target': 'test.user@example.com', 'detail': ''}]


In [70]:
def try_it(label, fn, *a):
    try: print("OK   ", label, "->", fn(*a))
    except ValueError as e: print("BLOCK", label, "->", e)

try_it("lock test user", AP.set_lock, "admin@local", "test.user@example.com", True)
try_it("unlock test user", AP.set_lock, "admin@local", "test.user@example.com", False)
try_it("lock yourself", AP.set_lock, "admin@local", "admin@local", True)
try_it("demote last admin", AP.set_role, "someone@else.com", "admin@local", "Staff", None)
try_it("delete without DELETE", AP.delete_user, "admin@local", "test.user@example.com", "")
try_it("delete test user", AP.delete_user, "admin@local", "test.user@example.com", "DELETE")
try_it("weak password", AP.add_user, "admin@local", "new.person@example.com", "NewPerson", "abc", "Auditor", None)

OK    lock test user -> test.user@example.com locked
OK    unlock test user -> test.user@example.com unlocked
BLOCK lock yourself -> You cannot lock your own account
BLOCK demote last admin -> The last remaining active Admin cannot be demoted
BLOCK delete without DELETE -> Type DELETE in the confirmation box to delete this user
OK    delete test user -> User test.user@example.com deleted
BLOCK weak password -> Password needs at least 8 characters


In [71]:
from google.colab import files
files.download("/content/app/admin_profile.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>